# Controles de calidad — Silver y Gold

Ejecutar tras `02_silver_transform` y (opcional) tras `03_gold_model`. Matriz: `docs/calidad_datos.md`.

In [ ]:
import sys
dbutils.widgets.text("repo_root", "", "Ruta Repo Databricks")
dbutils.widgets.text("batch_id", "", "Batch ID para log ops")
repo_root = dbutils.widgets.get("repo_root").rstrip("/")
if repo_root:
    sys.path.insert(0, f"{repo_root}/src")

from ips_analytics.config import DEFAULT_CONFIG
dbutils.widgets.dropdown("include_gold", "false", ["true", "false"], "Checks Gold (post Fase 4)")

from ips_analytics.quality.runner import (
    assert_quality_passed,
    run_pipeline_quality_checks,
)

In [ ]:
batch_id = dbutils.widgets.get("batch_id").strip() or "quality_manual"
include_gold = dbutils.widgets.get("include_gold") == "true"
report = run_pipeline_quality_checks(
    spark,
    batch_id=batch_id,
    include_bronze=True,
    include_silver=True,
    include_gold=include_gold,
    persist_ops=True,
)
for line in report.summary_lines():
    print(line)
assert_quality_passed(report)

In [ ]:
catalog = DEFAULT_CONFIG.catalog
try:
    spark.table(f"{catalog}.silver.rejects").groupBy("_entity", "_reject_reason").count().show(truncate=False)
except Exception as ex:
    print("Sin tabla rejects:", ex)